# 02 — Split dataset and save split files

This notebook loads your processed 4-channel image tensor and condition-name labels, then saves a fixed train/validation/test split.

Default split:

- train: 60%
- validation: 20%
- test: 20%

Outputs are saved under:

```text
<PROJECT_ROOT>/Output/splits/
    X_train.npy
    X_val.npy
    X_test.npy
    y_train_names.npy
    y_val_names.npy
    y_test_names.npy
    y_train_idx.npy
    y_val_idx.npy
    y_test_idx.npy
    class_names.npy
    label_mapping.json
    split_indices.npz
```

In [1]:
from pathlib import Path
import pickle
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Change this path if your shared folder is different.
PROJECT_ROOT = Path(r"..")
OUTPUT_DIR = PROJECT_ROOT / "output"
SPLIT_DIR = OUTPUT_DIR / "splits"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = PROJECT_ROOT / "large_condition_source_4ch_100x100.npy"
TARGET_FILE = PROJECT_ROOT / "large_condition_target_names.npy"

RANDOM_SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20  # validation proportion of the full dataset

print("Output folder:", OUTPUT_DIR)
print("Split folder:", SPLIT_DIR)
print("Source file:", SOURCE_FILE)
print("Target file:", TARGET_FILE)


Output folder: ..\output
Split folder: ..\output\splits
Source file: ..\large_condition_source_4ch_100x100.npy
Target file: ..\large_condition_target_names.npy


In [2]:
def load_array(path):
    path = Path(path)

    if path.suffix.lower() in [".p", ".pkl", ".pickle"]:
        with open(path, "rb") as f:
            return pickle.load(f)

    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=True)

    raise ValueError(f"Unsupported file type: {path}")


X = load_array(SOURCE_FILE)
y = load_array(TARGET_FILE)

X = np.asarray(X, dtype=np.float32)
y_names = np.asarray(y).astype(str)

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("y shape:", y_names.shape)
print("Example label:", y_names[0])
print("X min/max before normalization:", float(np.min(X)), float(np.max(X)))

if X.ndim != 4:
    raise ValueError(f"Expected X shape (N, 4, H, W), but got {X.shape}")

if X.shape[1] != 4:
    raise ValueError(f"Expected 4 channels at axis 1, but got shape {X.shape}")

if len(X) != len(y_names):
    raise ValueError(f"X and y length mismatch: {len(X)} vs {len(y_names)}")


X shape: (137390, 4, 100, 100)
X dtype: float32
y shape: (137390,)
Example label: Novobiocin_2x
X min/max before normalization: 0.0 1.0


In [3]:
np.unique(y)

array(['A22_05x', 'A22_2x', 'Ampicillin_05x', 'Ampicillin_2x', 'Blank',
       'Chloramphenicol_05x', 'Chloramphenicol_2x', 'Control',
       'Gramacidin_05x', 'Gramacidin_2x', 'Gramacidin_5x',
       'Kanamycin_05x', 'Kanamycin_2x', 'Mecillinam_05x', 'Mecillinam_2x',
       'Mecillinam_5x', 'Naladixic_acid_05x', 'Naladixic_acid_2x',
       'Nisin_05x', 'Nisin_2x', 'Nisin_5x', 'Novobiocin_05x',
       'Novobiocin_2x', 'PMB_05x', 'PMB_2x', 'PMB_5x', 'Rifampicin_05x',
       'Rifampicin_2x', 'Rifampicin_5x', 'Triclosan_05x', 'Triclosan_2x',
       'Triclosan_5x'], dtype=object)

In [4]:
# Normalize image values to [0, 1] for PyTorch ImageNet transforms.
# If your data is already normalized, this will leave it unchanged.

X = np.asarray(X, dtype=np.float32)

if np.max(X) > 1.5:
    X = X / np.max(X)

X = np.clip(X, 0.0, 1.0)

print("X min/max after normalization:", float(np.min(X)), float(np.max(X)))
print("X shape:", X.shape)


X min/max after normalization: 0.0 1.0
X shape: (137390, 4, 100, 100)


In [5]:
class_names = np.asarray(sorted(np.unique(y_names).tolist()))
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}

y_idx = np.asarray([class_to_idx[name] for name in y_names], dtype=np.int64)

print("Number of classes:", len(class_names))
print("Class distribution:")
display(pd.Series(y_names).value_counts().sort_index().to_frame("count"))

print("Class mapping:")
for i, name in enumerate(class_names):
    print(i, "->", name)

Number of classes: 32
Class distribution:


,count
A22_05x,6602
A22_2x,6061
Ampicillin_05x,1958
Ampicillin_2x,1967
Blank,1
Chloramphenicol_05x,1805
Chloramphenicol_2x,2357
Control,11202
Gramacidin_05x,6197
Gramacidin_2x,1662


Class mapping:
0 -> A22_05x
1 -> A22_2x
2 -> Ampicillin_05x
3 -> Ampicillin_2x
4 -> Blank
5 -> Chloramphenicol_05x
6 -> Chloramphenicol_2x
7 -> Control
8 -> Gramacidin_05x
9 -> Gramacidin_2x
10 -> Gramacidin_5x
11 -> Kanamycin_05x
12 -> Kanamycin_2x
13 -> Mecillinam_05x
14 -> Mecillinam_2x
15 -> Mecillinam_5x
16 -> Naladixic_acid_05x
17 -> Naladixic_acid_2x
18 -> Nisin_05x
19 -> Nisin_2x
20 -> Nisin_5x
21 -> Novobiocin_05x
22 -> Novobiocin_2x
23 -> PMB_05x
24 -> PMB_2x
25 -> PMB_5x
26 -> Rifampicin_05x
27 -> Rifampicin_2x
28 -> Rifampicin_5x
29 -> Triclosan_05x
30 -> Triclosan_2x
31 -> Triclosan_5x


In [6]:
indices = np.arange(len(y_names))

class_counts = pd.Series(y_names).value_counts()
can_stratify = class_counts.min() >= 3

if can_stratify:
    stratify_all = y_idx
    print("Using stratified train/val/test split.")
else:
    stratify_all = None
    print("Warning: at least one class has fewer than 3 samples. Using non-stratified split.")

trainval_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=stratify_all,
)

val_relative_size = VAL_SIZE / (1.0 - TEST_SIZE)

if can_stratify:
    stratify_trainval = y_idx[trainval_idx]
else:
    stratify_trainval = None

train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=val_relative_size,
    random_state=RANDOM_SEED,
    stratify=stratify_trainval,
)

print("Train size:", len(train_idx))
print("Validation size:", len(val_idx))
print("Test size:", len(test_idx))
print("Total:", len(train_idx) + len(val_idx) + len(test_idx))


Train size: 82434
Validation size: 27478
Test size: 27478
Total: 137390


In [7]:
splits = {
    "train": train_idx,
    "val": val_idx,
    "test": test_idx,
}

for split_name, split_idx in splits.items():
    np.save(SPLIT_DIR / f"X_{split_name}.npy", X[split_idx])
    np.save(SPLIT_DIR / f"y_{split_name}_names.npy", y[split_idx])
    np.save(SPLIT_DIR / f"y_{split_name}_idx.npy", y[split_idx])

np.save(SPLIT_DIR / "class_names.npy", class_names)

np.savez(
    SPLIT_DIR / "split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
)

with open(SPLIT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "class_to_idx": class_to_idx,
            "idx_to_class": {str(k): v for k, v in idx_to_class.items()},
            "random_seed": RANDOM_SEED,
            "test_size": TEST_SIZE,
            "val_size": VAL_SIZE,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved split files to:", SPLIT_DIR)
print("Files:")
for path in sorted(SPLIT_DIR.glob("*")):
    print("-", path.name)


Saved split files to: ..\output\splits
Files:
- class_names.npy
- label_mapping.json
- split_indices.npz
- X_test.npy
- X_train.npy
- X_val.npy
- y_test_idx.npy
- y_test_names.npy
- y_train_idx.npy
- y_train_names.npy
- y_val_idx.npy
- y_val_names.npy
